# Code-Graph Tier-2 — Cross-Artifact & External-Symbol Fusion Design Spec

> **Companion to** `2026-06-04-code-graph-ontology-tier0-design.ipynb` (T-Box / schema-of-schema) and
> `2026-06-05-code-graph-binding-precision-tier1-design.ipynb` (binding precision).
>
> **Status:** draft (2026-06-05). **Premise, earned not assumed:** Tier-1 is *complete* — every
> deterministic bind arm is **0 cross-crate / 0 cross-language** across two independent corpora
> (SPUR, Rust-dominant; `anywidget`, Python+JS). The residual resolution gap is **not** a precision
> defect another gate can close. It is the **closed-world assumption itself**: the graph binds an edge
> only by name-matching against symbols it *owns*, so every call into stdlib / third-party crates / npm,
> and every import whose disambiguation needs a module path, is silently dropped.
>
> **Tier-2 gives the graph a model of the symbols it does not own, and the paths that connect them.**
>
> Every *evidence* cell below queries the same live DuckDB analyst artifact the `code_*` MCP tools read —
> the design is grounded in the current graph, not in recollection.

In [2]:
# --- Live evidence harness: read-only connection to the SPUR analyst graph artifact ---
# Every "evidence" cell below queries this same DuckDB artifact the code_* MCP tools use.
import duckdb, pandas as pd

ANALYST_DB = "/Volumes/Projects/spur/.spur/analyst.duckdb"
con = duckdb.connect(ANALYST_DB, read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).fetchdf()

# Freshness + scale — Tier-2 reasoning is only valid against a known artifact (expect v12 + JS extractor).
q("""SELECT graph_content_hash[1:12] AS artifact, manifest_version[1:12] AS manifest,
            node_count, resolved_edge_count FROM _meta""")

,artifact,manifest,node_count,resolved_edge_count
0,9b1fef993779,ad80ce9ad284,50439,98002


## 1. Why Tier-2 — the wall Tier-1 cannot scale

Tier-1 spent six resolver versions (v6→v12) making bare-name binds *safe*: crate-scope gates, the
language-family gate, the per-language builtin denylist, constructor & relational candidate filters.
It worked — precision is at the ceiling. But precision only governs the edges the graph **can** bind.
The **recall ceiling** is set by something the gate methodology cannot touch: a closed world.

The resolver's entire model of "what exists" is *the workspace symbol table*. Three structural
consequences follow, and they are the whole of Tier-2:

1. **External calls are invisible.** A call to `tokio::spawn`, `Vec::push`, `serde_json::from_str`,
   or `React.createElement` has no workspace target, so it is dropped. The unresolved-call mass is
   dominated by std/iterator/3p mechanics, not by missed workspace edges.
2. **Imports can't disambiguate without paths.** The extractor stores an import as its *bare final
   segment* (`RelationKind`, not `crate::extract::schema::RelationKind`). When that bare name has
   several workspace definitions, the import is dropped — even though the full `use` path names
   exactly one.
3. **Cross-crate recall is unlicensed.** Because imports don't resolve to symbols, a cross-crate call
   can only bind through the `scope_match` qualifier heuristic; everything else stays unresolved.

The next cell quantifies the ceiling. Read it as *the budget Tier-2 is trying to recover*.

In [3]:
# The resolution ceiling across the binding relations — the Tier-2 motivation.
# 'pct' is the share of edges of each relation that bind to a workspace target.
q("""
WITH r AS (SELECT relation, count(*) resolved FROM edges GROUP BY relation),
     u AS (SELECT relation, count(*) unresolved FROM edges_unresolved GROUP BY relation)
SELECT coalesce(r.relation,u.relation) AS relation,
       coalesce(resolved,0)   AS resolved,
       coalesce(unresolved,0) AS unresolved,
       round(100.0*coalesce(resolved,0)/nullif(coalesce(resolved,0)+coalesce(unresolved,0),0),1) AS pct_resolved
FROM r FULL OUTER JOIN u ON r.relation=u.relation
WHERE coalesce(r.relation,u.relation) IN ('calls','imports','constructs','extends','implements','references')
ORDER BY unresolved DESC
""")

,relation,resolved,unresolved,pct_resolved
0,calls,22385,87235,20.4
1,imports,4961,9686,33.9
2,constructs,985,1636,37.6
3,implements,286,300,48.8
4,extends,3,75,3.8
5,references,108,4,96.4


**Reading the ceiling (live v12 + JS-extractor artifact):** `calls` resolve at **~20%**
(22,385 / 109,620), `imports` at **~34%** (4,961 / 14,647), `constructs` at ~38%. These are not bugs —
they are the closed world. The ~87k unresolved calls are overwhelmingly **std / iterator / enum-variant
mechanics** (`new`, `Some`, `expect`, `clone`, `unwrap`, `Ok`, `iter`, `len`), confirmed in §3. Recall
will not move until the graph can name what lives *outside* the workspace.

### The three Tier-2 pillars

```mermaid
graph TD
    P1["Pillar 1 — Import-path resolution<br/>(the keystone)"]
    P2["Pillar 2 — External-symbol modeling<br/>(stdlib / 3p crates / npm)"]
    P3["Pillar 3 — Import-licensed cross-crate recall<br/>(Frontier B)"]
    P1 -->|"resolved import paths<br/>license cross-crate binds"| P3
    P2 -->|"external symbol table<br/>absorbs the unresolved mass"| P3
    P1 -->|"a use-path that leaves<br/>the workspace = an external ref"| P2
    classDef key fill:#1f6feb,color:#fff;
    class P1 key;
```

**Pillar 1 is the keystone**: capturing and resolving full import paths both (a) disambiguates the
in-workspace imports the bare-name resolver drops, and (b) is the *router* that tells Pillar 2 which
unresolved references point at external packages. Pillar 3 (Frontier B) is the recall payoff that
only becomes safe once imports resolve to symbols.

## 2. Pillar 1 — Import-path resolution (the keystone)

**The defect:** the extractor records an import by its *bare final segment*. `use crate::extract::schema::RelationKind`
becomes `target_label = "RelationKind"`; the path that disambiguates it is thrown away. So
`import_resolution_candidates` can only match bare name → workspace symbol, and must drop anything
ambiguous. The next cell decomposes the 9,686 unresolved imports to size each sub-population.

In [4]:
# Decompose the unresolved imports by recoverability. The bare-name resolver can only ever
# reach the 'unique workspace def' slice; the 'ambiguous' mass needs the full use-path.
q("""
WITH defcount AS (
  SELECT entity_name, count(*) n_defs
  FROM nodes WHERE file_path LIKE 'crates/%' GROUP BY entity_name
)
SELECT
  CASE WHEN dc.entity_name IS NULL THEN '1. external (no workspace def) -> Pillar 2'
       WHEN dc.n_defs = 1      THEN '2. unique workspace def (bare-name recoverable now)'
       ELSE '3. ambiguous bare name (needs full use-path -> Pillar 1)' END AS bucket,
  count(*) AS unresolved_imports
FROM edges_unresolved eu
LEFT JOIN defcount dc ON dc.entity_name = eu.target_label
WHERE eu.relation = 'imports'
GROUP BY bucket ORDER BY bucket
""")

,bucket,unresolved_imports
0,1. external (no workspace def) -> Pillar 2,5666
1,2. unique workspace def (bare-name recoverable...,81
2,3. ambiguous bare name (needs full use-path ->...,3939


**Reading it:** ~5,666 imports are **external** (no workspace def — they route to Pillar 2); only **81**
are bare-name-unique (a marginal `singleton-import` cleanup, ~0.08% of edges — not worth a standalone
merge); **3,939** are **ambiguous bare names** that the current resolver *structurally cannot* bind. The
3,939 are the prize, and they are unreachable without paths.

### Pillar 1 design

- **Extractor:** capture the **full import path** as a first-class field on the import edge
  (`import_path = "crate::extract::schema::RelationKind"`), in addition to the bare `target_label`.
  This is a `.scm`/extraction change → **`EXTRACTOR_VERSION` bump** and a new schema field (coordinate
  with the Tier-0 T-Box; likely `SCHEMA_VERSION` touch).
- **Resolver:** a **module-path resolver** that walks the path against the file/module tree —
  `crate::`, `super::`, `self::`, package-relative (`./x`, `../y`) for JS/TS, and dotted module paths
  for Python — and binds to the unique symbol the *path* names, not the bare segment.
- **Re-export following:** resolve `pub use a::B` chains so an import of the re-exported name lands on
  the original definition (a bounded transitive walk, cycle-guarded).
- **Safety:** path resolution is *more* precise than bare-name (it can only bind what the path names),
  so it strictly raises recall without new phantom risk; the language-family gate still applies to the
  final bind. Single-language corpora ⇒ predictable golden churn (newly-resolved imports only).

## 3. Pillar 2 — External-symbol modeling

The largest unresolved population by far is **calls into symbols the workspace does not define**. Today
they vanish: no node, no edge, no provenance. The next cell shows the head of that mass — it is almost
entirely stdlib / iterator / enum-variant / smart-pointer mechanics.

In [5]:
# The external-call mass: top unresolved call labels. These are the closed-world wall —
# std/iterator/enum/smart-pointer surface, not missed workspace edges.
q("""
SELECT target_label, count(*) AS sites
FROM edges_unresolved
WHERE relation = 'calls'
GROUP BY target_label
ORDER BY sites DESC
LIMIT 15
""")

,target_label,sites
0,new,4477
1,Some,3355
2,expect,2480
3,to_string,2461
4,clone,2384
5,into,2221
6,unwrap,1815
7,Ok,1776
8,map,1755
9,iter,1730


**Reading it:** `new` (4,477), `Some`/`Ok` (enum variants), `expect`/`unwrap`/`clone`/`to_string`/`into`
(std methods), `iter`/`map`/`len` (iterator mechanics). This is exactly the surface the per-language
**builtin denylists** (v10) currently suppress with a hard-coded list — a heuristic crutch. An external
model **retires the crutch**: instead of *denying* `clone`, the graph *names* `std::clone::Clone::clone`.

### Pillar 2 design

- **External symbol node kind:** introduce `NodeKind::External` (or an `is_external` provenance flag)
  for symbols the workspace references but does not define. Sourced from: (a) imports whose path leaves
  the workspace (Pillar 1 routes these), (b) a curated **stdlib/prelude manifest** per language, (c)
  dependency manifests (`Cargo.toml`, `package.json`, `pyproject.toml`) for package-level attribution.
- **Provenance tier:** every external node carries `origin ∈ {stdlib, dependency, unknown}` and a
  package id, so a call to `tokio::spawn` resolves to an external node tagged `dependency:tokio` — a
  real edge with honest confidence, not a dropped one.
- **Retire the denylists:** once `clone`/`new`/`Some` resolve to external/stdlib nodes, the v10
  per-language `*_BUILTIN_METHODS` denylists become redundant and can be deleted — the model replaces
  the heuristic. (This is the principled end-state the v10 plan explicitly anticipated.)
- **Scope discipline:** external nodes are **leaves** — the graph models *that* a symbol is external and
  *which package* owns it, not the package's internals. Fusing external package *bodies* is out of scope
  (that is a registry/indexing problem, not a graph one).

## 4. Pillar 3 — Import-licensed cross-crate recall (Frontier B)

Tier-1 deliberately **forbids** cross-crate bare-name binds (the crate-safety gate) because, without
evidence, a name match across crates is a coin-flip. A resolved import path *is* that evidence: if file
`A` imports `crate_b::Widget`, then a bare `Widget(...)` call in `A` may safely bind to `crate_b`'s
`Widget`. Pillar 1 supplies the license; Pillar 3 spends it. The next cell sizes the licensable pool —
unresolved calls whose bare label uniquely names a workspace symbol in a *different* crate.

In [6]:
# Frontier B licensable pool: unresolved calls whose bare label uniquely names a workspace
# function/method that lives in a DIFFERENT crate than the call site. These are the binds a
# resolved-import licensing pass could safely recover (today forbidden by the crate-safety gate).
q("""
WITH fdef AS (  -- workspace callables that are globally unique by name
  SELECT entity_name,
         any_value(regexp_extract(file_path,'crates/([^/]+)/',1)) AS def_crate,
         count(*) n_defs
  FROM nodes
  WHERE symbol_kind IN ('function','method') AND file_path LIKE 'crates/%'
  GROUP BY entity_name
  HAVING count(*) = 1
)
SELECT count(*) AS licensable_cross_crate_calls
FROM edges_unresolved eu
JOIN nodes src ON src.stable_symbol_id = eu.source_stable_id
JOIN fdef f     ON f.entity_name = eu.target_label
WHERE eu.relation = 'calls'
  AND src.file_path LIKE 'crates/%'
  AND regexp_extract(src.file_path,'crates/([^/]+)/',1) <> f.def_crate
""")

,licensable_cross_crate_calls
0,4933


**Reading it:** this count is the upper bound on Frontier B recall — cross-crate calls that a resolved
import would license to a unique workspace target (a multiple of the 81 same-crate-only imports from
Pillar 1, because cross-crate is exactly the case Tier-1 forbids without evidence).

### Pillar 3 design

- **License source:** the resolved import set from Pillar 1. For a call site, the in-scope imports of its
  file define the set of crates/modules whose symbols are bindable by bare name.
- **Bind rule:** a bare call `f(...)` binds cross-crate **iff** the file imports a path resolving to a
  unique `f`; stamp `bind_method = "import_licensed"` for provenance and so the rebind drop-guard
  recognizes it (same pattern as `method_crate_singleton`/`constructs_type_singleton`).
- **Precision invariant preserved:** no import license ⇒ no cross-crate bind (the Tier-1 gate still holds
  for the unlicensed majority). Tier-2 *widens* recall only where evidence exists; it never loosens the
  gate globally.
- **Dependency on Pillar 1 is hard:** B cannot ship before import-path resolution — without resolved
  imports there is no license to spend. This fixes the sequencing.

## 5. Sequencing & the Tier-1 → Tier-2 boundary

```mermaid
graph LR
    T1["Tier-1 COMPLETE<br/>v6-v12 precision gates<br/>0 cross-crate / 0 cross-lang"]
    Clite["C-lite (optional)<br/>81 singleton imports<br/>bare-name, marginal"]
    P1["Pillar 1<br/>import-path resolution<br/>(EXTRACTOR + SCHEMA bump)"]
    P2["Pillar 2<br/>external-symbol model<br/>(new NodeKind + manifest)"]
    P3["Pillar 3 / Frontier B<br/>import-licensed recall<br/>(bind_method=import_licensed)"]
    Retire["retire v10 denylists<br/>(model replaces heuristic)"]
    T1 --> Clite
    T1 --> P1
    P1 --> P3
    P1 --> P2
    P2 --> Retire
    P2 --> P3
```

**The boundary, stated precisely.** Tier-1 = *bind correctly among symbols the graph owns*. Tier-2 =
*model the symbols and paths the graph does not own, then bind across that boundary with evidence*. The
tell is the resolver primitive: Tier-1 matches **bare names**; Tier-2 matches **paths** and introduces
**non-workspace nodes**. The moment a change needs an import path, a re-export walk, a dependency
manifest, or an external node, it is Tier-2.

**Recommended order:** Pillar 1 first (it is the keystone and the router for Pillar 2), then Pillar 2
(absorbs the unresolved mass + retires the v10 denylists), then Pillar 3/B (the recall payoff, gated on
Pillar 1). C-lite (81 imports) is an optional pre-Tier-2 cleanup, only if bundled cheaply.

## 6. Scope boundary — what is **not** Tier-2

- **External package *bodies*.** Tier-2 models *that* a symbol is external and *which package* owns it
  (a leaf node + provenance). Indexing the internals of `tokio`/`react`/`numpy` is a registry/multi-repo
  problem → **Tier-3+**.
- **Type inference / receiver-type resolution.** Discriminating an overloaded bare method by inferring
  its receiver's type is a *type-system* capability, not a path/symbol one. (Tier-1's `scope_match`
  already covers the qualifier-explicit case; the inference case is deferred.)
- **Whole-program dataflow / taint / effects.** Tier-2 is still a *structural* graph (symbols + edges).
  Semantic flow analysis is a later tier.
- **Cross-repository fusion.** One workspace at a time. Linking SPUR's graph to anywidget's is Tier-3+.
- **Re-litigating precision.** Tier-1 is closed; Tier-2 must **preserve** every precision invariant
  (0 cross-crate on ungated arms, 0 cross-language, qualifier-safe `scope_match`). Any Tier-2 bind that
  would create a cross-language edge or an unlicensed cross-crate edge is a bug, not a feature.

## 7. Carry-over invariants & acceptance criteria

**Realization contract (from Tier-0) still binds.** Every Tier-2 edge must be reproducible from the
artifact + the SHA-pinned manifest. New provenance (`import_path`, `is_external`/`origin`,
`bind_method="import_licensed"`) participates in `current_manifest_version()`, so any extractor/schema/
resolver change **bumps the matching version constant** and forces a clean re-derive. External nodes are
content-addressed like workspace nodes; they are *additive* and never mutate a workspace symbol's id.

**Acceptance criteria for the Tier-2 epic (each pillar lands behind these):**

- [ ] **Precision unregressed:** post-change, gated bind arms remain **0 cross-crate**, all arms remain
      **0 cross-language**, on both the SPUR and `anywidget` graphs (the two-corpus check used throughout
      Tier-1).
- [ ] **Pillar 1:** import-path captured + module-path resolver + re-export following; the 3,939 ambiguous
      imports measurably drop; goldens re-blessed with only newly-resolved import edges changing.
- [ ] **Pillar 2:** `External` node kind + provenance tier; the top unresolved-call mass (`new`, `clone`,
      `Some`…) resolves to external/stdlib nodes; the v10 per-language denylists are deleted with no
      precision regression.
- [ ] **Pillar 3 / B:** `import_licensed` cross-crate binds appear only where a resolved import licenses
      them; the rebind drop-guard recognizes the new `bind_method`; no unlicensed cross-crate bind appears.
- [ ] **Versioning:** `EXTRACTOR_VERSION` (Pillar 1 capture, Pillar 2 nodes), `SCHEMA_VERSION` (new fields),
      `RESOLVER_VERSION` (Pillars 1+3 binding) bumped as touched; `incremental_ingest.rs` stays untouched.

**Next artifact:** decompose this spec into a Pillar-1 implementation plan
(`docs/superpowers/plans/…-import-path-resolution.md`) — the keystone — before any Tier-2 code.

---
*Grounded against the live v12 + JS-extractor analyst artifact; re-run every code cell to refresh the
evidence before acting on this spec.*